# 02 · Señales de contratación

Indicadores para **priorizar revisión**. Ninguno prueba irregularidad: hay razones legítimas para una oferta única, una
contratación directa o un proveedor dominante. Mirá primero `01_calidad`.

Montos: nominales salvo que cargues IPC (`python -m monitor_licitaciones ipc --csv ...`). Solo ARS.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from monitor_licitaciones.consultas import Base

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
db = Base()   # usa MLC_DSN; conexión de solo lectura
print(db.valor("SELECT current_database()"), "| parser", db.valor("SELECT max(parser_version) FROM bo.acto"))

In [ ]:
hay_ipc = db.valor("SELECT count(*) FROM ref.ipc") > 0
print("Pesos constantes:", "sí" if hay_ipc else "NO — los montos de distintos años no son comparables")

## Panorama

In [ ]:
pan = db.df("""SELECT anio, count(DISTINCT codigo_proceso) AS procesos, count(DISTINCT cuit) AS proveedores,
                     sum(monto_asignado) FILTER (WHERE moneda='ARS') AS monto_nominal,
                     sum(monto_constante) AS monto_constante
              FROM analitica.ganador GROUP BY anio ORDER BY anio""")
display(pan)
pan.plot(x="anio", y=["procesos", "proveedores"], marker="o", figsize=(8, 3), title="Procesos y proveedores por año")
plt.tight_layout(); plt.show()

## 1. Competencia: oferta única
Solo organismos con al menos 20 procesos con dato. El dato de cantidad de ofertas aparece en una minoría de los textos:
mirá `procesos_con_dato`.

In [ ]:
comp = db.df("""SELECT organismo, anio, procesos, procesos_con_dato, oferta_unica, pct_oferta_unica, promedio_ofertas
               FROM analitica.s_competencia WHERE procesos_con_dato >= 20
               ORDER BY pct_oferta_unica DESC NULLS LAST""")
comp.head(20)

## 2. Concentración de proveedores (HHI)
HHI: 0 = competencia atomizada; > 2.500 = alta concentración (umbral usual de defensa de la competencia).
Se excluyen organismo-año con menos de 5 proveedores para no sobreinterpretar.

In [ ]:
conc = db.df("""SELECT c.*, e.razon_social AS proveedor_principal
               FROM analitica.s_concentracion c LEFT JOIN ref.empresa e ON e.cuit = c.cuit_principal
               WHERE c.proveedores >= 5 ORDER BY c.hhi DESC""")
display(conc.head(20))
if len(conc):
    conc["hhi"].astype(float).plot.hist(bins=30, figsize=(8, 3), title="Distribución del HHI (organismo-año)")
    plt.axvline(2500, ls="--", c="k"); plt.tight_layout(); plt.show()

## 3. Procedimientos de excepción
Peso de contrataciones directas (CDI) y menores (CME) por organismo.

In [ ]:
proc = db.df("""SELECT organismo, anio, procesos, directas, menores, pct_excepcion_cantidad, pct_excepcion_monto_nominal
               FROM analitica.s_procedimiento WHERE procesos >= 20 ORDER BY pct_excepcion_cantidad DESC""")
proc.head(20)

## 4. Posible fraccionamiento
3+ contrataciones menores/directas al mismo proveedor, mismo organismo, en 90 días. `supera_tope_acumulado` solo se
calcula si cargaste los topes en `ref.umbral_procedimiento`.

In [ ]:
fr = db.df("""SELECT f.organismo, f.cuit, e.razon_social, max(f.n_90d) AS max_en_90_dias,
                     sum(f.monto_asignado) AS monto_total, bool_or(f.supera_tope_acumulado) AS supera_tope,
                     min(f.fecha) AS desde, max(f.fecha) AS hasta
              FROM analitica.s_fraccionamiento f LEFT JOIN ref.empresa e ON e.cuit = f.cuit
              GROUP BY 1, 2, 3 ORDER BY max_en_90_dias DESC, monto_total DESC NULLS LAST""")
fr.head(25)

## 5. Empresas recién constituidas al ganar

In [ ]:
db.df("SELECT * FROM analitica.s_empresa_nueva ORDER BY dias_desde_constitucion, monto_asignado DESC NULLS LAST")

## 6. Domicilio fiscal compartido en el mismo acto
Depende de que ambas firmas tengan CUIT resuelto y domicilio en el padrón.

In [ ]:
db.df("SELECT * FROM analitica.s_domicilio_compartido ORDER BY fecha_publicacion DESC")

## 7. Prórrogas, ampliaciones y modificaciones

In [ ]:
db.df("SELECT * FROM analitica.s_modificaciones ORDER BY n_prorrogas + n_ampliaciones + n_modificaciones DESC LIMIT 25")

## 8. Proveedores para mirar primero
Cuenta en cuántas señales aparece cada proveedor. Es un **orden de revisión**, no un puntaje de riesgo.

In [ ]:
prov = db.df("""
WITH s AS (
  SELECT cuit, 'fraccionamiento' AS senal FROM analitica.s_fraccionamiento
  UNION SELECT cuit, 'empresa_nueva' FROM analitica.s_empresa_nueva
  UNION SELECT cuit_1, 'domicilio_compartido' FROM analitica.s_domicilio_compartido
  UNION SELECT cuit_2, 'domicilio_compartido' FROM analitica.s_domicilio_compartido
  UNION SELECT cuit_principal, 'principal_en_organismo_concentrado' FROM analitica.s_concentracion WHERE hhi > 2500 AND proveedores >= 5
)
SELECT p.cuit, p.razon_social, p.procesos_ganados, p.organismos, p.monto_nominal_ars, p.pct_oferta_unica, p.pct_excepcion,
       count(DISTINCT s.senal) AS n_senales, string_agg(DISTINCT s.senal, ', ') AS senales
FROM analitica.proveedor p JOIN s USING (cuit)
GROUP BY p.cuit, p.razon_social, p.procesos_ganados, p.organismos, p.monto_nominal_ars, p.pct_oferta_unica, p.pct_excepcion
ORDER BY n_senales DESC, p.monto_nominal_ars DESC NULLS LAST""")
prov.head(30)